# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "clients.csv"
RAW_TXN = "transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [42]:
CLIENTS_PATH = "clients.csv"
TXN_PATH     = "transactions.csv"

for path, enc in [(CLIENTS_PATH, "utf-8"), (TXN_PATH, "cp1251")]:
    with open(path, encoding=enc) as f:
        for _ in range(5):
            print(repr(f.readline()))
    print()

'client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in\n'
'C002260,1993-11-03,Ж,ЕКАТЕРИНБУРГ,Массовый,Базовый,Дебетовая,23.09.2022,21.06.2023,Не указан,86200.0,Да,2024-03,,00509,Да\n'
'C004875,1962-08-22,Ж,Москва,Зарплатный,Базовый,Кредитная,04.06.2017,07.05.2021,Зарплата в нашем банке,125900.0,Да,2024-12,,00611,Да\n'
'C004474,1970-02-15,М,Казань,Массовый,Базовый,Дебетовая,20.12.2012,05.04.2016,Зарплата в нашем банке,51700.0,Да,2024-06,2,00829,Нет\n'
'C005867,1986-01-23,Ж,Казань,Зарплатный,Серебро,Дебетовая,25.10.2019,23.10.2023,Зарплата в другом банке,90600.0,Нет,,,00119,Да\n'

'txn_id;client_id;txn_datetime;amount;currency;category;mcc_code;channel;status\n'
'990007;C001947;2023-11-23 09:00:42;823,86;RUB;Продукты;5411;Карта;успешно\n'
'990164;C005366;2023-11-24 14:58:42;68,41;RUB;Транспорт;4111;Карта;успешно\n'
'990056;C005023;2023-11-2

Файл clients.csv читается как UTF-8, разделителем является запятая, дробная часть отделена точкой. 
Даты представлены в разных форматах birth_date - Y-m-d, registration_date и card_open_date - d.m.Y. app_last_login - месяц. client_id - идентификатор,содержащий буквы и branch_code - строка, city - разный регистр. 
transactions.csv читается как cp1251, разделителем является точка с запятой, дробной - запятая. Формат даты - Y-m-d, в txn_datetime гдето-то указана дата без времени, txn_id, client_id, mcc_code - строки идентификаторы.

In [43]:
clients_raw = pd.read_csv(
    CLIENTS_PATH,
    encoding="utf-8",      
    sep=",",
    decimal=".",
    dtype={
        "client_id": "str",     
        "branch_code": "str"
    },
)
clients = clients_raw.copy()
txn_raw = pd.read_csv(
    TXN_PATH,
    encoding="cp1251",
    sep=";",
    decimal=",",                 
    dtype={
        "txn_id": "str",
        "client_id": "str",   
        "mcc_code": "str"
    },
)
txn = txn_raw.copy()

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [59]:
def overview(data):
    return pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "n_unique": data.nunique(),
        "n_missing": data.isna().sum(),
        "missing_%": (data.isna().mean() * 100).round(2),
        "example": data.iloc[0],
    })
print("Clients")
print('Размер:', clients.shape)
display(overview(clients))
print("Transactions")
print('Размер:', txn.shape)
display(overview(txn))

Clients
Размер: (5694, 16)


,dtype,n_unique,n_missing,missing_%,example
client_id,str,5639,0,0.00,C002260
birth_date,datetime64[us],4808,0,0.00,1993-11-03 00:00:00
gender,category,2,0,0.00,Ж
city,category,6,0,0.00,Екатеринбург
segment,category,4,0,0.00,Массовый
loyalty_tier,category,4,0,0.00,Базовый
card_type,category,3,0,0.00,Дебетовая
registration_date,datetime64[us],3235,0,0.00,2022-09-23 00:00:00
card_open_date,datetime64[us],2642,0,0.00,2023-06-21 00:00:00
income_source,category,3,0,0.00,Не указан


Transactions
Размер: (624130, 9)


,dtype,n_unique,n_missing,missing_%,example
txn_id,str,622630,0,0.0,990007
client_id,str,5991,0,0.0,C001947
txn_datetime,datetime64[us],567137,0,0.0,2023-11-23 09:00:42
amount,float64,322443,0,0.0,823.86
currency,category,3,0,0.0,RUB
category,category,9,0,0.0,Продукты
mcc_code,category,9,0,0.0,5411
channel,category,4,0,0.0,Карта
status,category,3,0,0.0,успешно


Изучив две таблицы выше, обращаем внимение:
Даты пока представлены в формате строк.
Во второй таблице нет пропусков ни по одному признаку. 
Наблюдается большое количество с признаков с малым количеством уникальных значений, их  тип надо будет изменить на category.
clients_id в таблице клиентов имеет меньше уникальных значений, чем строк, что говорит о возможных дублях или наличии нескольких разных строк для одного клиента, это надо проверить. (то же самое можно сказать о txn_id)
client_id в таблице транзакций  наличие нескольких транзакций у одного клиента логично, однако количестко оригинальный айди клиента больше, чем в таблице клиентов, что говорит о серьезном нарушении целостности данных.
category имеет 12 уникальных значений, когда в описании данных указано 9.
Единицы наблюдения client_id и transaction_id.

### 2.2. Дубли

In [45]:
print("Полных дубликатов строк в clients:", clients.duplicated().sum())
print("повторы client_id =", len(clients) - clients['client_id'].nunique())
print("Полных дубликатов строк в txn:", txn.duplicated().sum())
print("повторы txn_id =", len(txn) - txn['txn_id'].nunique())

Полных дубликатов строк в clients: 40
повторы client_id = 55
Полных дубликатов строк в txn: 1308
повторы txn_id = 1500


Количество полных дубликатов меньше количества не уникальных значений единиц наблюдения в обоих таблицах

### 2.3. Соответствие данных описанию полей из задания

In [46]:
print(txn["category"].value_counts().to_string(), "\n")
domains = {
    "clients": {
        "segment": {"Массовый", "Зарплатный", "Премиальный", "Private"},
        "loyalty_tier": {"Базовый", "Серебро", "Золото", "Платина"},
        "card_type": {"Дебетовая", "Кредитная", "Премиальная"},
    },
    "txn": {
        "channel": {"Карта", "Токен", "СБП", "Онлайн-банк"},
        "status": {"успешно", "отклонено", "возврат"},
    }
}
for tbl_name, tbl in [("clients", clients), ("txn", txn)]:
    for col, exp in domains[tbl_name].items():
        act = set(tbl[col].dropna().unique())
        if act != exp: print(f"{tbl_name}.{col}: Плохо")
        else: print(f"{tbl_name}.{col}: OK")

print("satisfaction_score: min", clients["satisfaction_score"].min(), 
      "| max", clients["satisfaction_score"].max())

txn_dates = pd.to_datetime(txn['txn_datetime'], format='mixed')
print(f"Период операций: {txn_dates.min().date()} .. {txn_dates.max().date()}")

category
Продукты               164942
Транспорт               72396
Одежда и обувь          64242
АЗС                     57338
Кафе и рестораны        48226
Развлечения и досуг     47045
Аптеки                  45461
Онлайн-сервисы          43503
Связь и интернет        29176
Связь и Интернет        19805
Кафе/рестораны          16036
кафе и рестораны        15960 

clients.segment: OK
clients.loyalty_tier: OK
clients.card_type: OK
txn.channel: OK
txn.status: OK
satisfaction_score: min 1.0 | max 5.0
Период операций: 2023-11-23 .. 2026-07-26


Как мы наблюдали выше количество уникальных значений category не совпало с заявленным, смотрим все уникальные значения и видим, что кафе и рестораны трижды записаны по разному, а связь и интернет дважды.
Период операций выходит за пределы заявленного

In [47]:
print(clients['city'].unique())

<StringArray>
[      'ЕКАТЕРИНБУРГ',             'Москва',             'Казань',       'Екатеринбург',          'г. Москва',    'Санкт-Петербург', 'г. Санкт-Петербург',
             'МОСКВА',        'Новосибирск',            ' москва',     'г.Екатеринбург',        'Н. Новгород',             'казань',    'Нижний Новгород',
                'СПб',    'Санкт Петербург',     'г. Новосибирск']
Length: 17, dtype: str


Смотрим на уникальные значения sity, видим, что проблема не только в регистре, одни и те же города записаны по разному, с буквой г. или без, через -, требуется нормализация

### 2.4. Связь между таблицами

In [48]:
ids_in_clients = set(clients['client_id'])
ids_in_txn = set(txn['client_id'])

orphans = ids_in_txn - ids_in_clients
print(f"1. Клиентов в txn, которых нет в clients: {len(orphans)}")
dormant = ids_in_clients - ids_in_txn
print(f"2. Клиентов в clients, у которых нет операций: {len(dormant)}")

temp_map = clients.drop_duplicates('client_id').set_index('client_id')['card_open_date']
card_dates = txn['client_id'].map(temp_map)
txn_dates = pd.to_datetime(txn['txn_datetime'], format='mixed')
card_dates = pd.to_datetime(card_dates, format='%d.%m.%Y')
print("3. Операций раньше открытия карты:", (txn_dates < card_dates).sum())

txn['txn_datetime'].str.len().value_counts()

1. Клиентов в txn, которых нет в clients: 361
2. Клиентов в clients, у которых нет операций: 9
3. Операций раньше открытия карты: 1


txn_datetime
19    574042
10     50088
Name: count, dtype: int64

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

В clients найдено 40 дублей (55 по client_id), в txn - 1308 (1500 по txn_id).Требует очистки, чтобы не использовать одни и те же данные дважды.
361 клиент совершает операции, но не существует, нарушена целостность данных. 9 клиентов есть в справочнике, но не имеют операций, надо проверить, возможно это просто новые клиенты, а не призраки. Найдена 1 операция до открытия карты. В sity встречаются разные записи одних  и тех  же, по количеству уникальных значений 17 понимаем, что признак категориальный, надо нормализовать регистры и вид записи, чтобы избежать дублей категорий.
В txn.category 12 значений вместо заявленных 9 из-за разного написания, нужна нормализация, без которой оценить вклад категорий не получится. 
Период совершения операций выходит за пределы заявленного, в txn_datetime встречаются даты со временем и без около 50 тысяч строк.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [49]:
na_before = clients[['birth_date', 'registration_date', 'card_open_date', 'app_last_login']].isna().sum()
clients['birth_date'] = pd.to_datetime(clients['birth_date'], format='%Y-%m-%d', errors='coerce')
clients['registration_date'] = pd.to_datetime(clients['registration_date'], format='%d.%m.%Y', errors='coerce')
clients['card_open_date'] = pd.to_datetime(clients['card_open_date'], format='%d.%m.%Y', errors='coerce')
clients['app_last_login'] = pd.to_datetime(clients['app_last_login'], format='%Y-%m', errors='coerce')

na_after = clients[['birth_date', 'registration_date', 'card_open_date', 'app_last_login']].isna().sum()
print("Новые NaTв clients:")
print(na_after - na_before)

na_before = txn['txn_datetime'].isna().sum()
txn['txn_datetime'] = pd.to_datetime(txn['txn_datetime'], format='mixed', errors='coerce')
na_after = txn['txn_datetime'].isna().sum()

print("txn_datetime ", na_after - na_before)

Новые NaTв clients:
birth_date           0
registration_date    0
card_open_date       0
app_last_login       0
dtype: int64
txn_datetime  0


### 3.2. Категориальные признаки

In [ ]:
original_cat = txn['category'].copy()
txn['category'] = txn['category'].replace({
    'Кафе/рестораны': 'Кафе и рестораны',
    'кафе и рестораны': 'Кафе и рестораны',
    'Связь и Интернет': 'Связь и интернет'
})
n_changed_cat = (original_cat != txn['category']).sum()
print(f"Строк в category изменено: {n_changed_cat}")
original_city = clients['city'].copy()
city_clean = clients['city'].str.lower().str.strip().str.replace(r'^г\.\s*', '', regex=True)

city_clean = city_clean.replace({
    'спб': 'санкт-петербург',
    'санкт петербург': 'санкт-петербург',
    'н. новгород': 'нижний новгород'
})
clients['city'] = city_clean.str.title()
n_changed_city = (original_city != clients['city']).sum()
print(f"Строк в city изменено: {n_changed_city}")

Строк в category изменено: 51801
Строк в city изменено: 1685


In [58]:
cat_cols_cli = ['gender', 'city', 'segment', 'loyalty_tier', 'card_type', 'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in']
clients[cat_cols_cli] = clients[cat_cols_cli].astype('category')

cat_cols_txn = ['currency', 'category', 'mcc_code', 'channel', 'status']
txn[cat_cols_txn] = txn[cat_cols_txn].astype('category')


### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [62]:

missing = clients.isna().mean() * 100
print("Доля пропусков (%):")
print(missing[missing > 0].round(2), "\n")

Доля пропусков (%):
monthly_income         8.85
app_last_login        37.95
satisfaction_score    59.54
dtype: float64 



In [ ]:
print("Пропуски в app_last_login по наличию приложения:")
print(clients.groupby('has_mobile_app')['app_last_login'].apply(lambda x: x.isna().mean() * 100).round(2), "\n")

Пропуски в app_last_login по наличию приложения:
has_mobile_app
Да       0.0
Нет    100.0
Name: app_last_login, dtype: float64 



Видим, что дата последнего входа в приложение пропущена исключительно у клиентов, не зарегестрированных в приложении, значит, пропуски логичны.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [51]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

In [52]:
new_logs = [
    {
        "field": "txn_datetime", "issue": "строковый формат (с временем и без)",
        "scope": "transactions", "n_rows": len(txn),
        "value_before": "str ('2023-11-23 09:00:42', '2023-11-26')", "action": "конвертация",
        "value_after": "datetime64", "rationale": "для корректного расчета периодов"
    },
    {
        "field": "txn_datetime, birth_date, registration_date, card_open_date, app_last_login", 
        "issue": "строковый формат на датах", "scope": "clients", "n_rows": len(clients),
        "value_before": "str ('1993-11-03', '23.09.2022', '2024-03')", "action": "конвертация",
        "value_after": "datetime64", "rationale": "для расчета возраста, стажа и давности входа"
    }
]

cleaning_log = pd.concat([cleaning_log, pd.DataFrame(new_logs)], ignore_index=True)

In [55]:
new_logs2 = [
    {
        "field": "category", 
        "issue": "разнобой в написании, 12 уникальных значений вместо 9", 
        "scope": "txn", 
        "n_rows": 51801,  
        "value_before": "'Кафе/рестораны', 'кафе и рестораны', 'Связь и Интернет'", 
        "action": "исправлено", 
        "value_after": "9 уникальных категорий", 
        "rationale": "устранение дублей категорий"
    },
    {
        "field": "city", 
        "issue": "разный регистр, приставки, сокращения (17 вариантов)", 
        "scope": "clients", 
        "n_rows": 1685, 
        "value_before": "'г. Москва', ' москва', 'СПб', 'ЕКАТЕРИНБУРГ'", 
        "action": "нормализация", 
        "value_after": "6 городов", 
        "rationale": "устранение дублей категорий"
    },
    {
        "field": "cat_cols (segment, city, category и др.)", 
        "issue": "неоптимальный тип данных для полей с малым числом уникальных значений", 
        "scope": "clients, txn", 
        "n_rows": len(clients) + len(txn), 
        "value_before": "str", 
        "action": "изменение типа", 
        "value_after": "category", 
        "rationale": "экономия оперативной памяти"
    }
]
cleaning_log = pd.concat([cleaning_log, pd.DataFrame(new_logs2)], ignore_index=True)

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [54]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")